In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_68_try_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 69 ###

# cell 69 optimized

# Define axis title and question strings
x_axis_title_cell81 = 'Frequency of TPU usage'
question_2022 = 'Approximately how many times have you used a TPU (tensor processing unit)?'
question_2019 = 'Have you ever used a TPU (tensor processing unit)?'

# Standardize 2019 responses in-place using a single mapping replace
responses_df_2019_cell10[question_2019] = responses_df_2019_cell10[question_2019].replace({
    '6-24 times': '6-25 times',
    '> 25 times': 'More than 25 times'
})

# Rename the 2019 column to match the 2022 question text
responses_df_2019_cell10.columns = (
    responses_df_2019_cell10.columns
    .str.replace(question_2019, question_2022, regex=False)
)

# Vectorized count-and-percent function using GPU-friendly groupby
def count_then_return_percent_81(df, col):
    counts = df.groupby(col, sort=True).size()                # GPU groupby
    return (counts * 100 / counts.sum()).round(1)            # GPU arithmetic and rounding

# Build a list of per-year DataFrames
dfs = []
for year, df in [
    ('2019', responses_df_2019_cell10),
    ('2020', responses_df_2020),
    ('2021', responses_df_2021),
    ('2022', responses_df_2022_cell10)
]:
    pct = count_then_return_percent_81(df, question_2022)
    tmp = pct.reset_index()
    tmp.columns = [x_axis_title_cell81, 'percentage']
    tmp['year'] = year
    # reorder columns
    dfs.append(tmp[['percentage', 'year', x_axis_title_cell81]])

# Concatenate and final sort entirely on GPU
df_combined_cell81 = pd.concat(dfs).sort_values(by=['year', 'percentage'], ascending=True)

df_combined_cell81.info()